In [ ]:
from pathlib import Path
import os
import re
import shutil
import subprocess
from collections import OrderedDict, Counter

import numpy as np
import pandas as pd
import torch

from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, r2_score, ndcg_score


# Main switches

MUTATION_LEVEL = "multi"           # "single" or "multi"

EMBEDDING_MODE = "original"        # "original" or "finetuned"
ZERO_SHOT_MODE = "default"         # "default" or "ensemble"

# Data preparation / feature extraction
RUN_PREPARE_DATA = True
RUN_COPY_PDBS = True
RUN_PATCH_REFERENCE_FILE = True

RUN_ESM2_EMBEDDINGS = True
RUN_PROTEINMPNN_RAW = True
RUN_PROTEINMPNN_PROCESSING = True
RUN_3D_COORDS = True
RUN_ESM2_ZERO_SHOT = True

# Benchmark
RUN_KERMUT_BENCHMARK = True
FORCE_RERUN_BENCHMARK = True

# If None, generated automatically
RUN_LABEL = None
OVERWRITE = True


# Validate switches
VALID_MUTATION_LEVELS = {"single", "multi"}
VALID_EMBEDDING_MODES = {"original", "finetuned"}
VALID_ZERO_SHOT_MODES = {"default", "ensemble"}

if MUTATION_LEVEL not in VALID_MUTATION_LEVELS:
    raise ValueError(f"MUTATION_LEVEL must be one of {VALID_MUTATION_LEVELS}")

if EMBEDDING_MODE not in VALID_EMBEDDING_MODES:
    raise ValueError(f"EMBEDDING_MODE must be one of {VALID_EMBEDDING_MODES}")

if ZERO_SHOT_MODE not in VALID_ZERO_SHOT_MODES:
    raise ValueError(f"ZERO_SHOT_MODE must be one of {VALID_ZERO_SHOT_MODES}")

if MUTATION_LEVEL == "multi" and ZERO_SHOT_MODE == "ensemble":
    print(
        "[WARN] ZERO_SHOT_MODE='ensemble' for multi mutants only works if you already "
        "created a Kermut kernel/config that points to your multi-mutant ensemble scores."
    )

if RUN_LABEL is None:
    RUN_LABEL = f"{MUTATION_LEVEL}_{EMBEDDING_MODE}_{ZERO_SHOT_MODE}"

print("MUTATION_LEVEL:", MUTATION_LEVEL)
print("EMBEDDING_MODE:", EMBEDDING_MODE)
print("ZERO_SHOT_MODE:", ZERO_SHOT_MODE)
print("RUN_LABEL:", RUN_LABEL)

MUTATION_LEVEL: multi
EMBEDDING_MODE: original
ZERO_SHOT_MODE: default
RUN_LABEL: multi_original_default


In [ ]:
# Global paths

KERMUT_REPO = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut"
)

KERMUT_DATA_DIR = KERMUT_REPO / "data"
KERMUT_OUTPUTS_DIR = KERMUT_REPO / "outputs"

FUSIONGP_DATA_BASE = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/FusionGP/data"
)

FINAL_DATA_DIR = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data"
)

FINAL_DATA_BASE = FINAL_DATA_DIR

PROTEINMPNN_DIR = KERMUT_REPO / "ProteinMPNN"

ESM_MODEL_FILE_ORIGINAL = KERMUT_REPO / "models" / "esm2_t33_650M_UR50D.pt"
ESM_CONTACT_FILE_ORIGINAL = KERMUT_REPO / "models" / "esm2_t33_650M_UR50D-contact-regression.pt"


# Output folders
OUT_BASE = FINAL_DATA_DIR / "Kermut" / "variant_runs" / RUN_LABEL
OUT_BASE.mkdir(parents=True, exist_ok=True)

LOG_DIR = OUT_BASE / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_OUTPUT_DIR = OUT_BASE / "predictions"
ARCHIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS_OUT_DIR = OUT_BASE / "metrics"
METRICS_OUT_DIR.mkdir(parents=True, exist_ok=True)

PREPARED_DIR = OUT_BASE / "prepared_csv"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DIR_OUT = OUT_BASE / "splits"
SPLIT_DIR_OUT.mkdir(parents=True, exist_ok=True)

METADATA_DIR = OUT_BASE / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

PDB_OUT_DIR = OUT_BASE / "pdbs"
PDB_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KERMUT_REPO:", KERMUT_REPO)
print("KERMUT_DATA_DIR:", KERMUT_DATA_DIR)
print("OUT_BASE:", OUT_BASE)

KERMUT_REPO: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut
KERMUT_DATA_DIR: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data
OUT_BASE: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/Kermut/variant_runs/multi_original_default


In [ ]:
# Kermut mode, directories and kernel

if MUTATION_LEVEL == "single":
    KERMUT_DATASET_CLI = "single"
    KERMUT_DATASET_OVERRIDE_KEY = "single"
    KERMUT_DATASET_OVERRIDE_PREFIX = ""

    KERMUT_CV_DIR = KERMUT_DATA_DIR / "cv_folds_singles_substitutions"
    KERMUT_EMB_DIR = KERMUT_DATA_DIR / "embeddings" / "substitutions_singles" / "ESM2"

elif MUTATION_LEVEL == "multi":
    KERMUT_DATASET_CLI = "multi"
    KERMUT_DATASET_OVERRIDE_KEY = "multi"

    # Important:
    # In this Kermut repo, "multi" is not a predefined top-level Hydra key.
    # Therefore we append it with "+".
    KERMUT_DATASET_OVERRIDE_PREFIX = "+"

    KERMUT_CV_DIR = KERMUT_DATA_DIR / "cv_folds_multiples_substitutions"
    KERMUT_EMB_DIR = KERMUT_DATA_DIR / "embeddings" / "substitutions_multiples" / "ESM2"

else:
    raise ValueError(f"Unknown MUTATION_LEVEL: {MUTATION_LEVEL}")


KERMUT_PDB_DIR = KERMUT_DATA_DIR / "structures" / "pdbs"
KERMUT_COORDS_DIR = KERMUT_DATA_DIR / "structures" / "coords"
KERMUT_RAW_MPNN_DIR = KERMUT_DATA_DIR / "conditional_probs" / "raw_ProteinMPNN_outputs"
KERMUT_PROC_MPNN_DIR = KERMUT_DATA_DIR / "conditional_probs" / "ProteinMPNN"
KERMUT_ZERO_SHOT_DIR = KERMUT_DATA_DIR / "zero_shot_fitness_predictions" / "ESM2" / "650M"

for p in [
    KERMUT_CV_DIR,
    KERMUT_EMB_DIR,
    KERMUT_PDB_DIR,
    KERMUT_COORDS_DIR,
    KERMUT_RAW_MPNN_DIR,
    KERMUT_PROC_MPNN_DIR,
    KERMUT_ZERO_SHOT_DIR,
    KERMUT_OUTPUTS_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)


# Kernel selection
if ZERO_SHOT_MODE == "default":
    KERNEL_NAME = "kermut"
elif ZERO_SHOT_MODE == "ensemble":
    KERNEL_NAME = "kermut_ensemble"
else:
    raise ValueError(f"Unknown ZERO_SHOT_MODE: {ZERO_SHOT_MODE}")

TEMP_PRED_DIR = KERMUT_REPO / "outputs" / "split_esm2" / KERNEL_NAME


def hydra_dataset_id_overrides(dms_id: str):
    """
    Builds the correct Hydra overrides for single/multi.

    single:
        single.use_id=true single.id=...

    multi:
        +multi.use_id=true +multi.id=...
    """
    prefix = KERMUT_DATASET_OVERRIDE_PREFIX
    key = KERMUT_DATASET_OVERRIDE_KEY

    return (
        f"{prefix}{key}.use_id=true "
        f"{prefix}{key}.id={dms_id}"
    )


print("KERMUT_DATASET_CLI:", KERMUT_DATASET_CLI)
print("KERMUT_DATASET_OVERRIDE_KEY:", KERMUT_DATASET_OVERRIDE_KEY)
print("KERMUT_DATASET_OVERRIDE_PREFIX:", KERMUT_DATASET_OVERRIDE_PREFIX)
print("KERMUT_CV_DIR:", KERMUT_CV_DIR)
print("KERMUT_EMB_DIR:", KERMUT_EMB_DIR)
print("KERNEL_NAME:", KERNEL_NAME)
print("TEMP_PRED_DIR:", TEMP_PRED_DIR)

KERMUT_DATASET_CLI: multi
KERMUT_DATASET_OVERRIDE_KEY: multi
KERMUT_DATASET_OVERRIDE_PREFIX: +
KERMUT_CV_DIR: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/cv_folds_multiples_substitutions
KERMUT_EMB_DIR: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/embeddings/substitutions_multiples/ESM2
KERNEL_NAME: kermut
TEMP_PRED_DIR: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/outputs/split_esm2/kermut


In [ ]:
# Dataset config

SINGLE_DATASETS = [
    {
        "name": "UBE4B",
        "source_id": "UBE4B",
        "DMS_id": "UBE4B_original",
        "split_dir": FINAL_DATA_BASE / "UBE4B",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/data/experimental_datasets/pdb/UBE4B.pdb"),
        "indexing": "auto",
        "finetuned_pth": FUSIONGP_DATA_BASE / "UBE4B/kermut_inputs/ESM2_UBE4B_cropped_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "GRB2",
        "source_id": "GRB2",
        "DMS_id": "GRB2_original",
        "split_dir": FINAL_DATA_BASE / "GRB2",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/data/experimental_datasets/pdb/GRB2.pdb"),
        "indexing": "auto",
        "finetuned_pth": FUSIONGP_DATA_BASE / "GRB2/kermut_inputs/ESM2_GRB2_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "PTEN_activity",
        "source_id": "PTEN_activity",
        "DMS_id": "PTEN_activity_original",
        "split_dir": FINAL_DATA_BASE / "PTEN_activity",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/data/experimental_datasets/pdb/PTEN.pdb"),
        "indexing": "auto",
        "finetuned_pth": FUSIONGP_DATA_BASE / "PTEN_activity/kermut_inputs/ESM2_PTEN_activity_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "PTEN_abundance",
        "source_id": "PTEN_abundance",
        "DMS_id": "PTEN_abundance_original",
        "split_dir": FINAL_DATA_BASE / "PTEN_abundance",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/data/experimental_datasets/pdb/PTEN.pdb"),
        "indexing": "auto",
        "finetuned_pth": FUSIONGP_DATA_BASE / "PTEN_abundance/kermut_inputs/ESM2_PTEN_abundance_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "Alpha-Amylase_expression",
        "source_id": "Alpha-Amylase_expression",
        "DMS_id": "Alpha-Amylase_expression_original",
        "split_dir": FINAL_DATA_BASE / "Alpha_Amylase_expression",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/data/tournament/alpha_amylase_wt.pdb"),
        "indexing": "auto",
        "finetuned_pth": FUSIONGP_DATA_BASE / "Alpha-Amylase_expression/kermut_inputs/ESM2_Alpha-Amylase_expression_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "Alpha-Amylase_thermostability",
        "source_id": "Alpha-Amylase_thermostability",
        "DMS_id": "Alpha-Amylase_thermostability_original",
        "split_dir": FINAL_DATA_BASE / "Alpha_Amylase_thermostability",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/data/tournament/alpha_amylase_wt.pdb"),
        "indexing": "auto",
        "finetuned_pth": FUSIONGP_DATA_BASE / "Alpha-Amylase_thermostability/kermut_inputs/ESM2_Alpha-Amylase_thermostability_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "Alpha-Amylase_specific_activity",
        "source_id": "Alpha-Amylase_specific_activity",
        "DMS_id": "Alpha-Amylase_specific_activity_original",
        "split_dir": FINAL_DATA_BASE / "Alpha_Amylase_activity",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/data/tournament/alpha_amylase_wt.pdb"),
        "indexing": "auto",
        "finetuned_pth": FUSIONGP_DATA_BASE / "Alpha-Amylase_specific_activity/kermut_inputs/ESM2_Alpha-Amylase_specific_activity_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "DLG4_abundance",
        "source_id": "DLG4_abundance",
        "DMS_id": "DLG4_abundance_original",
        "split_dir": FINAL_DATA_BASE / "DLG4_abundance",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/DLG4.pdb"),
        "indexing": "0-based",
        "finetuned_pth": FUSIONGP_DATA_BASE / "DLG4_abundance/kermut_inputs/ESM2_DLG4_abundance_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "DLG4_binding",
        "source_id": "DLG4_binding",
        "DMS_id": "DLG4_binding_original",
        "split_dir": FINAL_DATA_BASE / "DLG4_binding",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/DLG4.pdb"),
        "indexing": "0-based",
        "finetuned_pth": FUSIONGP_DATA_BASE / "DLG4_binding/kermut_inputs/ESM2_DLG4_binding_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
]


MULTI_DATASETS = [
    {
        "name": "UBE4B_multi",
        "source_id": "UBE4B_multi",
        "DMS_id": "UBE4B_multi",
        "split_dir": FINAL_DATA_BASE / "UBE4B" / "multi",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/data/experimental_datasets/pdb/UBE4B.pdb"),
        "indexing": "0-based",
        "finetuned_pth": FUSIONGP_DATA_BASE / "UBE4B/kermut_inputs/ESM2_UBE4B_multi_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "DLG4_abundance_multi",
        "source_id": "DLG4_abundance_multi",
        "DMS_id": "DLG4_abundance_multi",
        "split_dir": FINAL_DATA_BASE / "DLG4_abundance" / "multi",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/DLG4.pdb"),
        "indexing": "0-based",
        "finetuned_pth": FUSIONGP_DATA_BASE / "DLG4_abundance/kermut_inputs/ESM2_DLG4_abundance_multi_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "DLG4_binding_multi",
        "source_id": "DLG4_binding_multi",
        "DMS_id": "DLG4_binding_multi",
        "split_dir": FINAL_DATA_BASE / "DLG4_binding" / "multi",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/test_data/DLG4.pdb"),
        "indexing": "0-based",
        "finetuned_pth": FUSIONGP_DATA_BASE / "DLG4_binding/kermut_inputs/ESM2_DLG4_binding_multi_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
    {
        "name": "GRB2_multi",
        "source_id": "GRB2_multi",
        "DMS_id": "GRB2_multi",
        "split_dir": FINAL_DATA_BASE / "GRB2" / "multi",
        "pdb_path": Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/data/experimental_datasets/pdb/GRB2.pdb"),
        "indexing": "0-based",
        "finetuned_pth": FUSIONGP_DATA_BASE / "GRB2/kermut_inputs/ESM2_GRB2_multi_650M_lr_5e-06_layers_0_bs_2_best_model.pth",
    },
]


DATASETS = SINGLE_DATASETS if MUTATION_LEVEL == "single" else MULTI_DATASETS

if EMBEDDING_MODE == "finetuned":
    missing = []

    for ds in DATASETS:
        if not Path(ds["finetuned_pth"]).exists():
            missing.append((ds["DMS_id"], ds["finetuned_pth"]))

    if missing:
        print("Missing fine-tuned .pth checkpoints:")

        for dms_id, path in missing:
            print(dms_id, "->", path)

        raise FileNotFoundError("Some fine-tuned .pth checkpoints are missing.")

print("Active datasets:")

for ds in DATASETS:
    print(ds["DMS_id"], "->", ds["split_dir"])

Active datasets:
UBE4B_multi -> /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/UBE4B/multi
DLG4_abundance_multi -> /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/DLG4_abundance/multi
DLG4_binding_multi -> /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/DLG4_binding/multi
GRB2_multi -> /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/GRB2/multi


In [ ]:
def run_cmd(cmd, cwd=None, env=None, log_file=None):
    print("\n" + "=" * 120)
    print("RUN:", cmd)
    print("=" * 120)

    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print(result.stdout)

    if log_file is not None:
        log_file = Path(log_file)
        log_file.parent.mkdir(parents=True, exist_ok=True)
        log_file.write_text(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n{cmd}\n\nLog:\n{result.stdout}"
        )

    return result


env = os.environ.copy()
env["PROTEINMPNN_DIR"] = str(PROTEINMPNN_DIR)
env["PYTHONPATH"] = str(KERMUT_REPO) + os.pathsep + env.get("PYTHONPATH", "")

print("PYTHONPATH:", env["PYTHONPATH"])
print("PROTEINMPNN_DIR:", env["PROTEINMPNN_DIR"])

PYTHONPATH: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut::/Users/kevinbauer/Masterarbeit/ProSST
PROTEINMPNN_DIR: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/ProteinMPNN


In [ ]:
# Split loading and mutation parsing

VARIANT_COL_CANDIDATES = [
    "variant",
    "mutant",
    "sample_id",
    "name",
]

SEQUENCE_COL_CANDIDATES = [
    "sequence",
    "mutated_sequence",
]

SCORE_COL_CANDIDATES = [
    "score",
    "DMS_score",
    "target",
    "label",
]

AA_PATTERN = re.compile(r"^([A-Z])(\d+)([A-Z])$")


def read_split_file(split_dir: Path, split: str) -> pd.DataFrame:
    candidates = [
        split_dir / f"{split}.tsv",
        split_dir / f"{split}.csv",
        split_dir / f"{split}_data.tsv",
        split_dir / f"{split}_data.csv",
        split_dir / f"{split}_multi.tsv",
        split_dir / f"{split}_multi.csv",
        split_dir / f"multi_{split}.tsv",
        split_dir / f"multi_{split}.csv",
    ]

    candidates += sorted(split_dir.glob(f"*{split}*.tsv"))
    candidates += sorted(split_dir.glob(f"*{split}*.csv"))

    seen = set()
    unique_candidates = []

    for path in candidates:
        if path not in seen:
            unique_candidates.append(path)
            seen.add(path)

    for path in unique_candidates:
        if path.exists():
            print(f"Reading {split} split from: {path}")

            if path.suffix == ".tsv":
                return pd.read_csv(path, sep="\t")

            return pd.read_csv(path)

    raise FileNotFoundError(
        f"No {split} file found in {split_dir}. Tried:\n"
        + "\n".join(str(p) for p in unique_candidates)
    )


def find_col(df: pd.DataFrame, candidates, required_name: str):
    cols = list(df.columns)
    lower_map = {c.lower(): c for c in cols}

    for c in candidates:
        if c in cols:
            return c

    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]

    raise ValueError(
        f"Could not find required column '{required_name}'. "
        f"Tried {candidates}. Available columns: {cols}"
    )


def split_mutation_string(mutant: str):
    """
    Converts a single or multi-mutant string into a list of substitutions.

    Accepted examples:
        A12G
        A12G:B45D
        A12G;B45D
        A12G,B45D
        A12G B45D
    """
    mutant = str(mutant).strip()

    if mutant == "" or mutant.lower() == "nan":
        raise ValueError("Empty mutant string.")

    parts = re.split(r"[:;,|\s]+", mutant)
    parts = [p.strip() for p in parts if p.strip()]

    if not parts:
        raise ValueError(f"Could not parse mutant string: {mutant}")

    return parts


def parse_single_substitution(sub: str):
    m = AA_PATTERN.fullmatch(str(sub).strip())

    if m is None:
        raise ValueError(f"Invalid substitution format: {sub}")

    wt, pos, mut = m.group(1), int(m.group(2)), m.group(3)

    return wt, pos, mut


def parse_variant(mutant: str):
    return [parse_single_substitution(p) for p in split_mutation_string(mutant)]


def canonicalize_mutant(mutant: str):
    """
    Canonical representation used internally:
        A12G:B45D:C67A
    Keeps the original numeric positions before PDB-local conversion.
    """
    parsed = parse_variant(mutant)

    return ":".join(f"{wt}{pos}{mut}" for wt, pos, mut in parsed)


def valid_number_of_mutations(n_mutations: int):
    """
    For single mode, exactly one substitution is required.
    For multi mode, all variants with at least two substitutions are accepted.
    """
    if MUTATION_LEVEL == "single":
        return n_mutations == 1

    if MUTATION_LEVEL == "multi":
        return n_mutations >= 2

    raise ValueError(f"Unknown MUTATION_LEVEL: {MUTATION_LEVEL}")

In [ ]:
# Apply mutation labels to sequences
def candidate_indices_from_position(pos: int, indexing: str):
    if indexing == "1-based":
        return [pos - 1]

    if indexing == "0-based":
        return [pos]

    if indexing == "auto":
        return [pos - 1, pos]

    raise ValueError("indexing must be one of: auto, 1-based, 0-based")


def infer_index_for_substitution(sequence: str, wt_aa: str, pos: int, mut_aa: str, indexing: str):
    observed = []

    for idx in candidate_indices_from_position(pos, indexing):
        if not (0 <= idx < len(sequence)):
            observed.append((idx, None))
            continue

        aa_at_pos = sequence[idx]
        observed.append((idx, aa_at_pos))

        if aa_at_pos == wt_aa or aa_at_pos == mut_aa:
            return idx

    raise ValueError(
        f"Could not match substitution {wt_aa}{pos}{mut_aa} to sequence. "
        f"Expected WT={wt_aa} or MUT={mut_aa}. "
        f"Tried indices/observed: {observed}. "
        f"Sequence length: {len(sequence)}"
    )


def infer_or_apply_variant(sequence: str, mutant: str, indexing: str = "auto"):
    """
    Handles both single and multi substitutions.

    Case 1:
        input sequence is WT or partly WT -> apply listed mutations.

    Case 2:
        input sequence is already mutated -> keep sequence if positions match mutant AA.
    """
    sequence = str(sequence).strip()
    parsed = parse_variant(mutant)

    seq_list = list(sequence)
    indices = []
    statuses = []

    for wt_aa, pos, mut_aa in parsed:
        idx = infer_index_for_substitution(
            sequence="".join(seq_list),
            wt_aa=wt_aa,
            pos=pos,
            mut_aa=mut_aa,
            indexing=indexing,
        )

        observed = seq_list[idx]
        indices.append(idx)

        if observed == wt_aa:
            seq_list[idx] = mut_aa
            statuses.append("mutation_applied")
        elif observed == mut_aa:
            statuses.append("already_mutated")
        else:
            raise ValueError(
                f"Unexpected residue for {wt_aa}{pos}{mut_aa}: "
                f"observed={observed}, idx={idx}"
            )

    if len(set(indices)) != len(indices):
        raise ValueError(f"Duplicate mutation positions in mutant: {mutant}")

    return "".join(seq_list), indices, "+".join(sorted(set(statuses)))


def load_dataset_splits(ds):
    dms_id = ds["DMS_id"]
    split_dir = Path(ds["split_dir"])
    indexing = ds.get("indexing", "auto")
    mutation_level_label = "exactly 1" if MUTATION_LEVEL == "single" else "at least 2"

    print("\n" + "=" * 100)
    print("Loading:", dms_id)
    print("Split dir:", split_dir)
    print("Indexing:", indexing)
    print("Expected number of substitutions:", mutation_level_label)

    dfs = []

    for split in ["train", "val", "test"]:
        raw = read_split_file(split_dir, split)

        variant_col = find_col(raw, VARIANT_COL_CANDIDATES, "variant")
        sequence_col = find_col(raw, SEQUENCE_COL_CANDIDATES, "sequence")
        score_col = find_col(raw, SCORE_COL_CANDIDATES, "score")

        df = raw[[variant_col, sequence_col, score_col]].copy()
        df.columns = ["mutant", "input_sequence", "DMS_score"]

        df["mutant"] = df["mutant"].astype(str).str.strip().map(canonicalize_mutant)
        df["input_sequence"] = df["input_sequence"].astype(str).str.strip()
        df["DMS_score"] = pd.to_numeric(df["DMS_score"], errors="coerce")
        df["split"] = split

        before = len(df)
        df = df.dropna(subset=["mutant", "input_sequence", "DMS_score"]).copy()
        after = len(df)

        if before != after:
            print(f"[WARN] {dms_id} {split}: removed {before - after} invalid rows.")

        n_mutations = df["mutant"].map(lambda x: len(parse_variant(x)))
        valid_n = n_mutations.map(valid_number_of_mutations)
        bad_n = ~valid_n

        df["n_mutations"] = n_mutations

        print(f"{split}: mutation-count distribution")
        print(df["n_mutations"].value_counts().sort_index())

        if bad_n.any():
            bad_path = METADATA_DIR / f"{dms_id}_{split}_wrong_number_mutations.tsv"

            df.loc[bad_n].to_csv(
                bad_path,
                sep="\t",
                index=False,
            )

            display(df.loc[bad_n].head(20))

            raise ValueError(
                f"{dms_id} {split}: found variants incompatible with MUTATION_LEVEL={MUTATION_LEVEL}. "
                f"For single, exactly one mutation is allowed. "
                f"For multi, at least two mutations are allowed. "
                f"Saved to {bad_path}"
            )

        mutated_sequences = []
        mutation_indices = []
        sequence_statuses = []
        failed_rows = []

        for row in df.itertuples(index=False):
            try:
                mutated_seq, indices, status = infer_or_apply_variant(
                    sequence=row.input_sequence,
                    mutant=row.mutant,
                    indexing=indexing,
                )

                mutated_sequences.append(mutated_seq)
                mutation_indices.append(":".join(map(str, indices)))
                sequence_statuses.append(status)

            except Exception as e:
                mutated_sequences.append(np.nan)
                mutation_indices.append(np.nan)
                sequence_statuses.append("failed")

                failed_rows.append({
                    "mutant": row.mutant,
                    "split": split,
                    "error": str(e),
                })

        df["mutated_sequence"] = mutated_sequences
        df["mutation_indices_0based"] = mutation_indices
        df["sequence_status"] = sequence_statuses

        if failed_rows:
            failed_df = pd.DataFrame(failed_rows)
            failed_path = METADATA_DIR / f"{dms_id}_{split}_failed_mutation_application.tsv"

            failed_df.to_csv(failed_path, sep="\t", index=False)
            display(failed_df.head(30))

            raise ValueError(
                f"{dms_id} {split}: failed to create mutated sequences. "
                f"Saved failures to {failed_path}"
            )

        dfs.append(df)

        print(f"{split}: {len(df)}")

    full = pd.concat(dfs, ignore_index=True)
    full["DMS_id"] = dms_id
    full["source_id"] = ds.get("source_id", dms_id)

    print("Sequence status counts:")
    print(full["sequence_status"].value_counts())
    print("Total:", len(full))
    print("Unique mutants:", full["mutant"].nunique())
    print("Unique sequences:", full["mutated_sequence"].nunique())
    print("Overall mutation-count distribution:")
    print(full["n_mutations"].value_counts().sort_index())

    return full

In [ ]:
AA3_TO_1 = {
    "ALA": "A",
    "ARG": "R",
    "ASN": "N",
    "ASP": "D",
    "CYS": "C",
    "GLN": "Q",
    "GLU": "E",
    "GLY": "G",
    "HIS": "H",
    "ILE": "I",
    "LEU": "L",
    "LYS": "K",
    "MET": "M",
    "PHE": "F",
    "PRO": "P",
    "SER": "S",
    "THR": "T",
    "TRP": "W",
    "TYR": "Y",
    "VAL": "V",
}


def extract_pdb_sequence_ca_only(pdb_path: Path, chain_id: str = None):
    pdb_path = Path(pdb_path)

    if not pdb_path.exists():
        raise FileNotFoundError(f"PDB not found: {pdb_path}")

    residues = []
    seen = set()

    with open(pdb_path, "r") as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue

            atom_name = line[12:16].strip()

            if atom_name != "CA":
                continue

            resname = line[17:20].strip()
            chain = line[21].strip()
            resseq = int(line[22:26])
            icode = line[26].strip()

            if chain_id is not None and chain != chain_id:
                continue

            key = (chain, resseq, icode)

            if key in seen:
                continue

            seen.add(key)

            if resname not in AA3_TO_1:
                continue

            residues.append({
                "chain": chain,
                "pdb_resseq": resseq,
                "icode": icode,
                "aa": AA3_TO_1[resname],
            })

    if not residues:
        raise ValueError(f"No protein CA residues found in PDB: {pdb_path}")

    residue_table = pd.DataFrame(residues)
    residue_table["local_pos_1based"] = np.arange(1, len(residue_table) + 1)

    sequence = "".join(residue_table["aa"].tolist())

    return sequence, residue_table


def infer_offset_to_pdb_sequence(df: pd.DataFrame, pdb_sequence: str, max_abs_offset: int = 10000):
    """
    Infers one constant offset:
        local_pos_1based = original_pos - offset

    Uses all substitutions from all variants.
    """
    candidates = Counter()

    for mutant in df["mutant"]:
        for wt_aa, original_pos, mut_aa in parse_variant(mutant):
            for local_pos_1based, aa in enumerate(pdb_sequence, start=1):
                if aa == wt_aa:
                    offset = original_pos - local_pos_1based

                    if abs(offset) <= max_abs_offset:
                        candidates[offset] += 1

    if not candidates:
        return None, 0

    return candidates.most_common(1)[0]


def validate_offset_to_pdb_sequence(df: pd.DataFrame, pdb_sequence: str, offset: int):
    n_total = 0
    n_match = 0
    mismatches = []

    for mutant in df["mutant"]:
        for wt_aa, original_pos, mut_aa in parse_variant(mutant):
            local_pos_1based = original_pos - offset
            idx = local_pos_1based - 1

            n_total += 1

            if not (0 <= idx < len(pdb_sequence)):
                mismatches.append({
                    "mutant": mutant,
                    "substitution": f"{wt_aa}{original_pos}{mut_aa}",
                    "original_pos": original_pos,
                    "local_pos_1based": local_pos_1based,
                    "pdb_sequence_length": len(pdb_sequence),
                    "problem": "position_out_of_range",
                })

                continue

            observed_wt = pdb_sequence[idx]

            if observed_wt == wt_aa:
                n_match += 1
            else:
                mismatches.append({
                    "mutant": mutant,
                    "substitution": f"{wt_aa}{original_pos}{mut_aa}",
                    "original_pos": original_pos,
                    "local_pos_1based": local_pos_1based,
                    "observed_wt": observed_wt,
                    "expected_wt": wt_aa,
                    "problem": "wt_mismatch",
                })

    return n_total, n_match, mismatches


def make_local_kermut_variant_and_sequence(mutant: str, pdb_sequence: str, offset: int):
    """
    Converts original mutation labels to PDB-local Kermut labels and creates
    the corresponding PDB-local mutated sequence.

    Example:
        original: N444G:A510V
        offset:   405
        local:    N39G:A105V
    """
    parsed = parse_variant(mutant)

    seq_list = list(pdb_sequence)
    local_parts = []
    local_positions = []

    for wt_aa, original_pos, mut_aa in parsed:
        local_pos_1based = original_pos - offset
        idx = local_pos_1based - 1

        if not (0 <= idx < len(pdb_sequence)):
            raise ValueError(
                f"{mutant}: local position out of range. "
                f"local_pos={local_pos_1based}, pdb_len={len(pdb_sequence)}"
            )

        if pdb_sequence[idx] != wt_aa:
            raise ValueError(
                f"{mutant}: WT mismatch at local position {local_pos_1based}. "
                f"PDB has {pdb_sequence[idx]}, mutant label expects {wt_aa}"
            )

        seq_list[idx] = mut_aa
        local_parts.append(f"{wt_aa}{local_pos_1based}{mut_aa}")
        local_positions.append(local_pos_1based)

    if len(set(local_positions)) != len(local_positions):
        raise ValueError(f"{mutant}: duplicate local mutation positions: {local_positions}")

    local_mutant = ":".join(local_parts)
    local_mutated_sequence = "".join(seq_list)

    return local_mutant, local_mutated_sequence, local_positions

In [ ]:
# Prepare Kermut input CSVs

loaded = {}

if RUN_PREPARE_DATA:
    for ds in DATASETS:
        dms_id = ds["DMS_id"]
        loaded[dms_id] = load_dataset_splits(ds)

prepared_summary = []

if RUN_PREPARE_DATA:
    for ds in DATASETS:
        dms_id = ds["DMS_id"]
        source_id = ds.get("source_id", dms_id)

        df = loaded[dms_id].copy()

        df_kermut = df[[
            "mutant",
            "DMS_score",
            "mutated_sequence",
            "split",
            "input_sequence",
            "mutation_indices_0based",
            "sequence_status",
            "n_mutations",
        ]].copy()

        df_kermut["original_mutant"] = df_kermut["mutant"]
        df_kermut["full_mutated_sequence"] = df_kermut["mutated_sequence"]

        pdb_sequence, pdb_residue_table = extract_pdb_sequence_ca_only(ds["pdb_path"])

        (METADATA_DIR / f"{dms_id}_pdb_sequence.txt").write_text(pdb_sequence)

        pdb_residue_table.to_csv(
            METADATA_DIR / f"{dms_id}_pdb_residue_table.tsv",
            sep="\t",
            index=False,
        )

        inferred_offset, support = infer_offset_to_pdb_sequence(df_kermut, pdb_sequence)

        if inferred_offset is None:
            raise ValueError(f"{dms_id}: Could not infer offset to PDB sequence.")

        n_total, n_match, mismatches = validate_offset_to_pdb_sequence(
            df_kermut,
            pdb_sequence,
            inferred_offset,
        )

        match_rate = n_match / n_total

        print("\n" + "=" * 100)
        print(dms_id)
        print("Source ID:", source_id)
        print("PDB sequence length:", len(pdb_sequence))
        print("Inferred PDB offset:", inferred_offset)
        print("Offset support:", support)
        print(f"Offset validation: {n_match}/{n_total} = {match_rate:.4f}")

        if match_rate < 0.95:
            mismatch_path = METADATA_DIR / f"{dms_id}_pdb_offset_mismatches.tsv"

            pd.DataFrame(mismatches).to_csv(
                mismatch_path,
                sep="\t",
                index=False,
            )

            display(pd.DataFrame(mismatches).head(30))

            raise ValueError(
                f"{dms_id}: PDB offset inference not stable enough "
                f"({n_match}/{n_total} = {match_rate:.4f}). "
                f"Saved mismatches to {mismatch_path}"
            )

        local_mutants = []
        local_mutated_sequences = []
        local_positions_all = []

        failed = []

        for row in df_kermut.itertuples(index=False):
            try:
                local_mutant, local_mutated_sequence, local_positions = make_local_kermut_variant_and_sequence(
                    mutant=row.original_mutant,
                    pdb_sequence=pdb_sequence,
                    offset=inferred_offset,
                )

                local_mutants.append(local_mutant)
                local_mutated_sequences.append(local_mutated_sequence)
                local_positions_all.append(":".join(map(str, local_positions)))

            except Exception as e:
                failed.append({
                    "original_mutant": row.original_mutant,
                    "error": str(e),
                })

        if failed:
            failed_path = METADATA_DIR / f"{dms_id}_local_kermut_conversion_failed.tsv"

            pd.DataFrame(failed).to_csv(
                failed_path,
                sep="\t",
                index=False,
            )

            display(pd.DataFrame(failed).head(30))

            raise ValueError(
                f"{dms_id}: failed to convert some variants to local Kermut format. "
                f"Saved to {failed_path}"
            )

        df_kermut["mutant"] = local_mutants
        df_kermut["mutated_sequence"] = local_mutated_sequences
        df_kermut["local_positions_1based"] = local_positions_all

        duplicated = df_kermut["mutant"].duplicated(keep=False)

        if duplicated.any():
            dup_path = METADATA_DIR / f"{dms_id}_duplicated_local_kermut_mutants.tsv"

            df_kermut.loc[duplicated].sort_values("mutant").to_csv(
                dup_path,
                sep="\t",
                index=False,
            )

            display(df_kermut.loc[duplicated].sort_values("mutant").head(50))

            raise ValueError(
                f"{dms_id}: duplicated local Kermut mutant labels found. "
                f"Saved to {dup_path}"
            )

        out_csv = PREPARED_DIR / f"{dms_id}.csv"
        df_kermut.to_csv(out_csv, index=False)

        df_benchmark = df_kermut[[
            "mutant",
            "DMS_score",
            "mutated_sequence",
            "split",
        ]].copy()

        df_benchmark = df_benchmark.rename(columns={"split": "split_esm2"})

        kermut_csv = KERMUT_CV_DIR / f"{dms_id}.csv"
        df_benchmark.to_csv(kermut_csv, index=False)

        for split in ["train", "val", "test"]:
            names = df_kermut.loc[df_kermut["split"] == split, "mutant"].to_numpy(dtype=object)
            np.save(SPLIT_DIR_OUT / f"{dms_id}_{split}_names.npy", names)

        split_table = df_kermut[[
            "mutant",
            "original_mutant",
            "split",
            "n_mutations",
            "local_positions_1based",
            "mutation_indices_0based",
            "sequence_status",
        ]].copy()

        split_table.to_csv(
            SPLIT_DIR_OUT / f"{dms_id}_splits.tsv",
            sep="\t",
            index=False,
        )

        prepared_summary.append({
            "source_id": source_id,
            "DMS_id": dms_id,
            "mutation_level": MUTATION_LEVEL,
            "n_total": len(df_kermut),
            "n_train": int((df_kermut["split"] == "train").sum()),
            "n_val": int((df_kermut["split"] == "val").sum()),
            "n_test": int((df_kermut["split"] == "test").sum()),
            "min_n_mutations": int(df_kermut["n_mutations"].min()),
            "max_n_mutations": int(df_kermut["n_mutations"].max()),
            "mean_n_mutations": float(df_kermut["n_mutations"].mean()),
            "pdb_sequence_length": len(pdb_sequence),
            "inferred_pdb_offset": int(inferred_offset),
            "offset_support": int(support),
            "offset_match_rate": float(match_rate),
            "unique_local_kermut_mutants": df_kermut["mutant"].nunique(),
            "unique_original_mutants": df_kermut["original_mutant"].nunique(),
            "unique_local_mutated_sequences": df_kermut["mutated_sequence"].nunique(),
            "target_seq_length": len(pdb_sequence),
            "target_seq": pdb_sequence,
            "prepared_csv": str(out_csv),
            "kermut_csv": str(kermut_csv),
        })

    summary_df = pd.DataFrame(prepared_summary)

    summary_df.to_csv(
        METADATA_DIR / "prepared_datasets_summary.tsv",
        sep="\t",
        index=False,
    )

    display(summary_df.drop(columns=["target_seq"]))

else:
    summary_df = pd.read_csv(
        METADATA_DIR / "prepared_datasets_summary.tsv",
        sep="\t",
    )

    prepared_summary = summary_df.to_dict("records")

    display(summary_df.drop(columns=["target_seq"], errors="ignore"))


Loading: UBE4B_multi
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/UBE4B/multi
Indexing: 0-based
Expected number of substitutions: at least 2
Reading train split from: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/UBE4B/multi/train.tsv
train: mutation-count distribution
n_mutations
2    41710
3    21249
4     5571
5     1189
6      196
7       26
8        6
9        1
Name: count, dtype: int64
train: 69948
Reading val split from: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/UBE4B/multi/val.tsv
val: mutation-count distribution
n_mutations
2     5182
3     2655
4      733
5      133
6       32
7        6
8        1
10       1
Name: count, dtype: int64
val: 8743
Reading test split from: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/UBE4B/multi/test.tsv
test: mutation-count distribution
n_mutations
2    5239
3    2668
4     673
5     132
6      28
7       3
8       1
Name: c

,source_id,DMS_id,mutation_level,n_total,n_train,n_val,n_test,min_n_mutations,max_n_mutations,mean_n_mutations,pdb_sequence_length,inferred_pdb_offset,offset_support,offset_match_rate,unique_local_kermut_mutants,unique_original_mutants,unique_local_mutated_sequences,target_seq_length,prepared_csv,kermut_csv
0,UBE4B_multi,UBE4B_multi,multi,87435,69948,8743,8744,2,10,2.527821,1173,-1,221020,1.0,87435,87435,87435,1173,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...
1,DLG4_abundance_multi,DLG4_abundance_multi,multi,5696,4556,570,570,2,2,2.000000,84,-1,11392,1.0,5696,5696,5696,84,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...
2,DLG4_binding_multi,DLG4_binding_multi,multi,6810,5448,681,681,2,2,2.000000,84,-1,13620,1.0,6810,6810,6810,84,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...
3,GRB2_multi,GRB2_multi,multi,62332,49865,6233,6234,2,2,2.000000,217,-1,124664,1.0,62332,62332,62332,217,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...


In [ ]:
# Split sanity check and PDB copying

for ds in DATASETS:
    dms_id = ds["DMS_id"]
    df = pd.read_csv(PREPARED_DIR / f"{dms_id}.csv")

    train = set(df.loc[df["split"] == "train", "mutant"])
    val = set(df.loc[df["split"] == "val", "mutant"])
    test = set(df.loc[df["split"] == "test", "mutant"])

    assert len(train & val) == 0, f"{dms_id}: train/val overlap"
    assert len(train & test) == 0, f"{dms_id}: train/test overlap"
    assert len(val & test) == 0, f"{dms_id}: val/test overlap"
    assert len(train | val | test) == len(df), f"{dms_id}: split union size mismatch"

    print(
        f"{dms_id}: OK | "
        f"train={len(train)} val={len(val)} test={len(test)} total={len(df)}"
    )


if RUN_COPY_PDBS:
    for ds in DATASETS:
        dms_id = ds["DMS_id"]
        src = Path(ds["pdb_path"])

        if not src.exists():
            raise FileNotFoundError(
                f"{dms_id}: PDB not found: {src}\n"
                "Please adjust pdb_path in DATASETS config."
            )

        dst_own = PDB_OUT_DIR / f"{dms_id}.pdb"
        dst_kermut = KERMUT_PDB_DIR / f"{dms_id}.pdb"

        if OVERWRITE or not dst_own.exists():
            shutil.copy2(src, dst_own)

        if OVERWRITE or not dst_kermut.exists():
            shutil.copy2(src, dst_kermut)

        print(f"{dms_id}: copied PDB")
        print("  own:   ", dst_own)
        print("  kermut:", dst_kermut)

else:
    print("RUN_COPY_PDBS=False -> skipped.")

UBE4B_multi: OK | train=69948 val=8743 test=8744 total=87435
DLG4_abundance_multi: OK | train=4556 val=570 test=570 total=5696
DLG4_binding_multi: OK | train=5448 val=681 test=681 total=6810
GRB2_multi: OK | train=49865 val=6233 test=6234 total=62332
UBE4B_multi: copied PDB
  own:    /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/Kermut/variant_runs/multi_original_default/pdbs/UBE4B_multi.pdb
  kermut: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/pdbs/UBE4B_multi.pdb
DLG4_abundance_multi: copied PDB
  own:    /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/Kermut/variant_runs/multi_original_default/pdbs/DLG4_abundance_multi.pdb
  kermut: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/pdbs/DLG4_abundance_multi.pdb
DLG4_binding_multi: copied PDB
  own:    /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/Kermut/variant_runs/multi_original_default/pdbs/D

In [ ]:
# Patch DMS_substitutions.csv

reference_file = KERMUT_DATA_DIR / "DMS_substitutions.csv"

if RUN_PATCH_REFERENCE_FILE:
    if not reference_file.exists():
        raise FileNotFoundError(
            f"Reference file not found: {reference_file}\n"
            "Please place ProteinGym DMS_substitutions.csv there first."
        )

    df_ref = pd.read_csv(reference_file)

    print("Reference rows before:", len(df_ref))
    print("Columns:", df_ref.columns.tolist()[:30])

    custom_ids = [ds["DMS_id"] for ds in DATASETS]

    df_ref_clean = df_ref[~df_ref["DMS_id"].isin(custom_ids)].copy()

    summary_by_id = {
        row["DMS_id"]: row
        for row in prepared_summary
    }

    custom_rows = []

    for ds in DATASETS:
        dms_id = ds["DMS_id"]
        info = summary_by_id[dms_id]

        row = {col: np.nan for col in df_ref_clean.columns}

        if "DMS_id" in row:
            row["DMS_id"] = dms_id

        if "UniProt_ID" in row:
            row["UniProt_ID"] = dms_id

        if "DMS_filename" in row:
            row["DMS_filename"] = f"{dms_id}.csv"

        if "raw_DMS_filename" in row:
            row["raw_DMS_filename"] = f"{dms_id}.csv"

        if "target_seq" in row:
            row["target_seq"] = info["target_seq"]

        if "target_sequence" in row:
            row["target_sequence"] = info["target_seq"]

        if "includes_multiple_mutants" in row:
            row["includes_multiple_mutants"] = MUTATION_LEVEL == "multi"

        if "DMS_total_number_mutants" in row:
            row["DMS_total_number_mutants"] = info["n_total"]

        if "DMS_number_single_mutants" in row:
            row["DMS_number_single_mutants"] = info["n_total"] if MUTATION_LEVEL == "single" else 0

        if "DMS_number_multiple_mutants" in row:
            row["DMS_number_multiple_mutants"] = 0 if MUTATION_LEVEL == "single" else info["n_total"]

        if "MSA_filename" in row:
            row["MSA_filename"] = np.nan

        if "pdb_file" in row:
            row["pdb_file"] = f"{dms_id}.pdb"

        custom_rows.append(row)

    df_custom = pd.DataFrame(custom_rows)
    df_ref_new = pd.concat([df_ref_clean, df_custom], ignore_index=True)

    backup_file = METADATA_DIR / "DMS_substitutions_before_custom_backup.csv"
    df_ref.to_csv(backup_file, index=False)

    df_ref_new.to_csv(reference_file, index=False)
    df_ref_new.to_csv(
        METADATA_DIR / "DMS_substitutions_with_custom_datasets.csv",
        index=False,
    )

    print("Reference rows after:", len(df_ref_new))

    display(df_custom)

else:
    print("RUN_PATCH_REFERENCE_FILE=False -> skipped.")

Reference rows before: 245
Columns: ['DMS_index', 'DMS_id', 'DMS_filename', 'UniProt_ID', 'taxon', 'source_organism', 'target_seq', 'seq_len', 'includes_multiple_mutants', 'DMS_total_number_mutants', 'DMS_number_single_mutants', 'DMS_number_multiple_mutants', 'DMS_binarization_cutoff', 'DMS_binarization_method', 'first_author', 'title', 'year', 'jo', 'region_mutated', 'molecule_name', 'selection_assay', 'selection_type', 'MSA_filename', 'MSA_start', 'MSA_end', 'MSA_len', 'MSA_bitscore', 'MSA_theta', 'MSA_num_seqs', 'MSA_perc_cov']
Reference rows after: 245


,DMS_index,DMS_id,DMS_filename,UniProt_ID,taxon,source_organism,target_seq,seq_len,includes_multiple_mutants,DMS_total_number_mutants,...,raw_DMS_filename,raw_DMS_phenotype_name,raw_DMS_directionality,raw_DMS_mutant_column,weight_file_name,pdb_file,pdb_range,ProteinGym_version,raw_mut_offset,coarse_selection_type
0,NaN,UBE4B_multi,UBE4B_multi.csv,UBE4B_multi,NaN,NaN,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,NaN,True,87435,...,UBE4B_multi.csv,NaN,NaN,NaN,NaN,UBE4B_multi.pdb,NaN,NaN,NaN,NaN
1,NaN,DLG4_abundance_multi,DLG4_abundance_multi.csv,DLG4_abundance_multi,NaN,NaN,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,NaN,True,5696,...,DLG4_abundance_multi.csv,NaN,NaN,NaN,NaN,DLG4_abundance_multi.pdb,NaN,NaN,NaN,NaN
2,NaN,DLG4_binding_multi,DLG4_binding_multi.csv,DLG4_binding_multi,NaN,NaN,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,NaN,True,6810,...,DLG4_binding_multi.csv,NaN,NaN,NaN,NaN,DLG4_binding_multi.pdb,NaN,NaN,NaN,NaN
3,NaN,GRB2_multi,GRB2_multi.csv,GRB2_multi,NaN,NaN,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...,NaN,True,62332,...,GRB2_multi.csv,NaN,NaN,NaN,NaN,GRB2_multi.pdb,NaN,NaN,NaN,NaN


In [ ]:
# Patch Kermut preprocessing scripts

files_to_patch = [
    KERMUT_REPO / "kermut" / "cmdline" / "preprocess_data" / "extract_esm2_embeddings.py",
    KERMUT_REPO / "kermut" / "cmdline" / "preprocess_data" / "extract_ProteinMPNN_probs.py",
    KERMUT_REPO / "kermut" / "cmdline" / "preprocess_data" / "extract_3d_coords.py",
    KERMUT_REPO / "kermut" / "cmdline" / "preprocess_data" / "extract_esm2_zero_shots.py",
]

for path in files_to_patch:
    if not path.exists():
        print("Missing file, skipping:", path)
        continue

    text = path.read_text()

    new_text = text.replace(
        'config_path="../hydra_configs"',
        'config_path="../../hydra_configs"',
    )

    if new_text != text:
        path.write_text(new_text)
        print(f"Patched Hydra config path: {path}")
    else:
        print(f"No Hydra path change needed: {path}")


# Patch extract_esm2_embeddings.py for model override
extract_file = KERMUT_REPO / "kermut" / "cmdline" / "preprocess_data" / "extract_esm2_embeddings.py"

if not extract_file.exists():
    raise FileNotFoundError(f"extract_esm2_embeddings.py not found: {extract_file}")

backup_file = extract_file.with_suffix(".py.before_finetuned_esm2_patch")

if not backup_file.exists():
    shutil.copy2(extract_file, backup_file)
    print("Backup written:", backup_file)
else:
    print("Backup already exists:", backup_file)

text = extract_file.read_text()

if "KERMUT_ESM2_MODEL_FILE" in text:
    print("ESM-2 model override patch already present.")
else:
    old = "model_path = Path(cfg.data.embedding.model_path)"
    new = 'model_path = Path(os.environ.get("KERMUT_ESM2_MODEL_FILE", cfg.data.embedding.model_path))'

    if old not in text:
        raise RuntimeError(
            "Could not find expected model_path line.\n"
            "Open extract_esm2_embeddings.py and search for cfg.data.embedding.model_path manually."
        )

    text = text.replace(old, new)

    if "import os" not in text.splitlines()[:20]:
        text = "import os\n" + text

    extract_file.write_text(text)
    print("Patched model_path successfully in:", extract_file)


print("\nPatch verification:")

for line in extract_file.read_text().splitlines():
    if "KERMUT_ESM2_MODEL_FILE" in line or "model_path =" in line:
        print(line)

No Hydra path change needed: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/kermut/cmdline/preprocess_data/extract_esm2_embeddings.py
No Hydra path change needed: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/kermut/cmdline/preprocess_data/extract_ProteinMPNN_probs.py
No Hydra path change needed: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/kermut/cmdline/preprocess_data/extract_3d_coords.py
No Hydra path change needed: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/kermut/cmdline/preprocess_data/extract_esm2_zero_shots.py
Backup already exists: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/kermut/cmdline/preprocess_data/extract_esm2_embeddings.py.before_finetuned_esm2_patch
ESM-2 model override patch already present.

Patch verification:
    model_path = Path(os.environ.get("KERMUT_ESM2_MODEL_FILE", cfg.data.embedding.model_path))


In [ ]:
# Convert fine-tuned .pth checkpoints to Kermut-compatible .pt files

if not ESM_MODEL_FILE_ORIGINAL.exists():
    raise FileNotFoundError(f"Missing original ESM-2 model: {ESM_MODEL_FILE_ORIGINAL}")

if not ESM_CONTACT_FILE_ORIGINAL.exists():
    raise FileNotFoundError(f"Missing original ESM-2 contact file: {ESM_CONTACT_FILE_ORIGINAL}")

CONVERTED_FINETUNED_ESM2_CHECKPOINTS = {}

if EMBEDDING_MODE == "finetuned":
    original_esm_data = torch.load(
        ESM_MODEL_FILE_ORIGINAL,
        map_location="cpu",
        weights_only=False,
    )

    if "model" not in original_esm_data:
        raise ValueError("Original ESM checkpoint has no 'model' key.")

    if "cfg" not in original_esm_data:
        raise ValueError(
            "Original ESM checkpoint has no 'cfg' key. "
            f"Available keys: {original_esm_data.keys()}"
        )

    for ds in DATASETS:
        dms_id = ds["DMS_id"]
        ft_path = Path(ds["finetuned_pth"])

        print("\n" + "=" * 100)
        print(dms_id)
        print("Input .pth:", ft_path)

        ft_state = torch.load(
            ft_path,
            map_location="cpu",
            weights_only=False,
        )

        if isinstance(ft_state, dict) and "model_state_dict" in ft_state:
            ft_state = ft_state["model_state_dict"]
        elif isinstance(ft_state, dict) and "state_dict" in ft_state:
            ft_state = ft_state["state_dict"]

        if not isinstance(ft_state, (dict, OrderedDict)):
            raise ValueError(f"{dms_id}: unexpected checkpoint type: {type(ft_state)}")

        converted_state = OrderedDict()
        skipped_keys = []

        for k, v in ft_state.items():
            k = str(k)

            if k.startswith("model."):
                new_k = k.replace("model.", "", 1)
                converted_state[new_k] = v
            elif k.startswith("fc."):
                skipped_keys.append(k)
            else:
                skipped_keys.append(k)

        print("Converted ESM keys:", len(converted_state))
        print("Skipped keys:", len(skipped_keys))

        if len(converted_state) == 0:
            raise ValueError(f"{dms_id}: no ESM keys converted. Check checkpoint format.")

        clean_task_name = dms_id.replace("_original", "")

        out_path = ft_path.with_name(
            f"esm2_{clean_task_name}_finetuned_for_kermut.pt"
        )

        esm_compatible = dict(original_esm_data)
        esm_compatible["model"] = converted_state

        torch.save(esm_compatible, out_path)

        contact_out = out_path.with_name(out_path.stem + "-contact-regression.pt")

        if contact_out.exists():
            print("Contact file already exists:", contact_out)
        else:
            try:
                os.symlink(ESM_CONTACT_FILE_ORIGINAL, contact_out)
                print("Created contact symlink:", contact_out)
            except OSError:
                shutil.copy2(ESM_CONTACT_FILE_ORIGINAL, contact_out)
                print("Copied contact file:", contact_out)

        CONVERTED_FINETUNED_ESM2_CHECKPOINTS[dms_id] = out_path

    print("\nConverted fine-tuned checkpoints:")

    for dms_id, path in CONVERTED_FINETUNED_ESM2_CHECKPOINTS.items():
        print(dms_id, "->", path)

else:
    print("EMBEDDING_MODE='original' -> skipping fine-tuned checkpoint conversion.")

EMBEDDING_MODE='original' -> skipping fine-tuned checkpoint conversion.


In [ ]:
# Extract ESM-2 embeddings

if RUN_ESM2_EMBEDDINGS:
    if not KERMUT_EMB_DIR.exists():
        KERMUT_EMB_DIR.mkdir(parents=True, exist_ok=True)

    embedding_mode_cli = "singles" if MUTATION_LEVEL == "single" else "multiples"

    for ds in DATASETS:
        dms_id = ds["DMS_id"]

        removed_files = []

        if FORCE_RERUN_BENCHMARK or OVERWRITE:
            for pattern in [f"{dms_id}*", f"*{dms_id}*"]:
                for file in KERMUT_EMB_DIR.glob(pattern):
                    if file.is_file():
                        file.unlink()
                        removed_files.append(file)

        print(f"{dms_id}: removed old ESM2 embedding files: {len(removed_files)}")

        env_ds = env.copy()

        if EMBEDDING_MODE == "finetuned":
            raise ValueError(
                "For the multi-mutant analysis, use EMBEDDING_MODE='original'. "
                "Fine-tuned multi Kermut is currently disabled intentionally."
            )
        else:
            env_ds.pop("KERMUT_ESM2_MODEL_FILE", None)
            embedding_label = "original ESM-2"
            log_suffix = "original"

        print("\n" + "=" * 120)
        print(dms_id)
        print("Using embeddings from:", embedding_label)
        print("Dataset CLI:", KERMUT_DATASET_CLI)
        print("Embedding mode CLI:", embedding_mode_cli)
        print("=" * 120)

        cmd = (
            "python -m kermut.cmdline.preprocess_data.extract_esm2_embeddings "
            f"dataset={KERMUT_DATASET_CLI} "
            f"data.embedding.mode={embedding_mode_cli} "
            f"{hydra_dataset_id_overrides(dms_id)}"
        )

        log_file = LOG_DIR / f"{dms_id}_extract_esm2_embeddings_{log_suffix}.log"

        run_cmd(
            cmd,
            cwd=KERMUT_REPO,
            env=env_ds,
            log_file=log_file,
        )

else:
    print("RUN_ESM2_EMBEDDINGS=False -> skipped.")

UBE4B_multi: removed old ESM2 embedding files: 0

UBE4B_multi
Using embeddings from: original ESM-2
Dataset CLI: multi
Embedding mode CLI: multiples

RUN: python -m kermut.cmdline.preprocess_data.extract_esm2_embeddings dataset=multi data.embedding.mode=multiples +multi.use_id=true +multi.id=UBE4B_multi

0it [00:00, ?it/s]--- Extracting embeddings for UBE4B_multi (1/1) ---

1it [38:57:41, 140261.06s/it]
1it [38:57:41, 140261.08s/it]

DLG4_abundance_multi: removed old ESM2 embedding files: 0

DLG4_abundance_multi
Using embeddings from: original ESM-2
Dataset CLI: multi
Embedding mode CLI: multiples

RUN: python -m kermut.cmdline.preprocess_data.extract_esm2_embeddings dataset=multi data.embedding.mode=multiples +multi.use_id=true +multi.id=DLG4_abundance_multi

0it [00:00, ?it/s]--- Extracting embeddings for DLG4_abundance_multi (1/1) ---

1it [07:52, 472.67s/it]
1it [07:52, 472.67s/it]

DLG4_binding_multi: removed old ESM2 embedding files: 0

DLG4_binding_multi
Using embeddings from: o

In [ ]:
# Check extracted ESM-2 embedding files

import h5py

embedding_check_rows = []

for ds in DATASETS:
    dms_id = ds["DMS_id"]

    emb_file = KERMUT_EMB_DIR / f"{dms_id}.h5"
    csv_file = KERMUT_CV_DIR / f"{dms_id}.csv"

    row = {
        "DMS_id": dms_id,
        "embedding_file": str(emb_file),
        "embedding_exists": emb_file.exists(),
        "csv_file": str(csv_file),
        "csv_exists": csv_file.exists(),
        "n_csv": np.nan,
        "n_embeddings": np.nan,
        "n_mutants_h5": np.nan,
    }

    if csv_file.exists():
        df_csv = pd.read_csv(csv_file)
        row["n_csv"] = len(df_csv)

    if emb_file.exists():
        with h5py.File(emb_file, "r") as h5f:
            print("\n" + "=" * 100)
            print(dms_id)
            print("H5 keys:", list(h5f.keys()))

            if "embeddings" in h5f:
                row["n_embeddings"] = len(h5f["embeddings"])
                print("n embeddings:", row["n_embeddings"])

            if "mutants" in h5f:
                row["n_mutants_h5"] = len(h5f["mutants"])
                print("n mutants:", row["n_mutants_h5"])

            print("n csv:", row["n_csv"])

    embedding_check_rows.append(row)

embedding_check_df = pd.DataFrame(embedding_check_rows)
display(embedding_check_df)


UBE4B_multi
H5 keys: ['embeddings', 'mutants']
n embeddings: 87435
n mutants: 87435
n csv: 87435

DLG4_abundance_multi
H5 keys: ['embeddings', 'mutants']
n embeddings: 5696
n mutants: 5696
n csv: 5696

DLG4_binding_multi
H5 keys: ['embeddings', 'mutants']
n embeddings: 6810
n mutants: 6810
n csv: 6810

GRB2_multi
H5 keys: ['embeddings', 'mutants']
n embeddings: 62332
n mutants: 62332
n csv: 62332


,DMS_id,embedding_file,embedding_exists,csv_file,csv_exists,n_csv,n_embeddings,n_mutants_h5
0,UBE4B_multi,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,True,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,True,87435,87435,87435
1,DLG4_abundance_multi,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,True,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,True,5696,5696,5696
2,DLG4_binding_multi,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,True,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,True,6810,6810,6810
3,GRB2_multi,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,True,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,True,62332,62332,62332


In [ ]:
# Raw ProteinMPNN, processed ProteinMPNN, 3D coordinates and ESM-2 zero-shot


# Copy PDB files into Kermut PDB directory

for ds in DATASETS:
    dms_id = ds["DMS_id"]

    pdb_path = Path(ds["pdb_path"])

    if not pdb_path.exists():
        raise FileNotFoundError(f"{dms_id}: PDB file not found: {pdb_path}")

    target_pdb = KERMUT_PDB_DIR / f"{dms_id}.pdb"
    target_pdb.parent.mkdir(parents=True, exist_ok=True)

    if FORCE_RERUN_BENCHMARK or OVERWRITE or not target_pdb.exists():
        target_pdb.write_bytes(pdb_path.read_bytes())
        print(f"{dms_id}: copied PDB to {target_pdb}")
    else:
        print(f"{dms_id}: PDB already exists at {target_pdb}")


# Generate raw ProteinMPNN outputs externally

if RUN_PROTEINMPNN_RAW:
    for ds in DATASETS:
        dms_id = ds["DMS_id"]

        print("\n" + "=" * 120)
        print(f"{dms_id}: raw ProteinMPNN generation")
        print("=" * 120)

        path_to_pdb = KERMUT_PDB_DIR / f"{dms_id}.pdb"
        output_dir = KERMUT_RAW_MPNN_DIR / dms_id / "proteinmpnn"
        expected_raw = output_dir / "conditional_probs_only" / f"{dms_id}.npz"

        if not path_to_pdb.exists():
            raise FileNotFoundError(f"{dms_id}: PDB not found: {path_to_pdb}")

        output_dir.mkdir(parents=True, exist_ok=True)

        if expected_raw.exists() and not (FORCE_RERUN_BENCHMARK or OVERWRITE):
            print(f"{dms_id}: raw ProteinMPNN already exists:")
            print(expected_raw)
            continue

        if expected_raw.exists() and (FORCE_RERUN_BENCHMARK or OVERWRITE):
            expected_raw.unlink()
            print("Removed old raw ProteinMPNN file:", expected_raw)

        cmd = (
            f'python "{PROTEINMPNN_DIR / "protein_mpnn_run.py"}" '
            f'--pdb_path "{path_to_pdb}" '
            '--save_score 1 '
            '--conditional_probs_only 1 '
            '--num_seq_per_target 10 '
            '--batch_size 1 '
            f'--out_folder "{output_dir}" '
            '--seed 37'
        )

        log_file = LOG_DIR / f"{dms_id}_proteinmpnn_raw.log"

        run_cmd(
            cmd,
            cwd=KERMUT_REPO,
            env=env,
            log_file=log_file,
        )

        if not expected_raw.exists():
            raise FileNotFoundError(
                f"{dms_id}: ProteinMPNN finished, but expected raw file was not found:\n"
                f"{expected_raw}\n"
                "Check the ProteinMPNN log file."
            )

        print(f"{dms_id}: raw ProteinMPNN written to {expected_raw}")

else:
    print("RUN_PROTEINMPNN_RAW=False -> skipped raw ProteinMPNN generation.")


# Process ProteinMPNN probabilities with Kermut

if RUN_PROTEINMPNN_PROCESSING:
    for ds in DATASETS:
        dms_id = ds["DMS_id"]

        print("\n" + "=" * 120)
        print(f"{dms_id}: ProteinMPNN probability processing")
        print("=" * 120)

        expected_raw = (
            KERMUT_RAW_MPNN_DIR
            / dms_id
            / "proteinmpnn"
            / "conditional_probs_only"
            / f"{dms_id}.npz"
        )

        if not expected_raw.exists():
            raise FileNotFoundError(
                f"{dms_id}: missing raw ProteinMPNN file:\n{expected_raw}\n"
                "Run RUN_PROTEINMPNN_RAW=True first."
            )

        if FORCE_RERUN_BENCHMARK or OVERWRITE:
            for pattern in [f"{dms_id}*", f"*{dms_id}*"]:
                for file in KERMUT_PROC_MPNN_DIR.glob(pattern):
                    if file.is_file():
                        file.unlink()
                        print("Removed old processed ProteinMPNN file:", file)

        cmd = (
            "python -m kermut.cmdline.preprocess_data.extract_ProteinMPNN_probs "
            f"dataset={KERMUT_DATASET_CLI} "
            f"{hydra_dataset_id_overrides(dms_id)}"
        )

        log_file = LOG_DIR / f"{dms_id}_extract_ProteinMPNN_probs.log"

        run_cmd(
            cmd,
            cwd=KERMUT_REPO,
            env=env,
            log_file=log_file,
        )

else:
    print("RUN_PROTEINMPNN_PROCESSING=False -> skipped ProteinMPNN probability processing.")


# Extract 3D coordinates

if RUN_3D_COORDS:
    for ds in DATASETS:
        dms_id = ds["DMS_id"]

        print("\n" + "=" * 120)
        print(f"{dms_id}: 3D coordinate extraction")
        print("=" * 120)

        if FORCE_RERUN_BENCHMARK or OVERWRITE:
            for pattern in [f"{dms_id}*", f"*{dms_id}*"]:
                for file in KERMUT_COORDS_DIR.glob(pattern):
                    if file.is_file():
                        file.unlink()
                        print("Removed old 3D coords file:", file)

        cmd = (
            "python -m kermut.cmdline.preprocess_data.extract_3d_coords "
            f"dataset={KERMUT_DATASET_CLI} "
            f"{hydra_dataset_id_overrides(dms_id)}"
        )

        log_file = LOG_DIR / f"{dms_id}_extract_3d_coords.log"

        run_cmd(
            cmd,
            cwd=KERMUT_REPO,
            env=env,
            log_file=log_file,
        )

else:
    print("RUN_3D_COORDS=False -> skipped 3D coordinate extraction.")

UBE4B_multi: copied PDB to /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/pdbs/UBE4B_multi.pdb
DLG4_abundance_multi: copied PDB to /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/pdbs/DLG4_abundance_multi.pdb
DLG4_binding_multi: copied PDB to /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/pdbs/DLG4_binding_multi.pdb
GRB2_multi: copied PDB to /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/pdbs/GRB2_multi.pdb

UBE4B_multi: raw ProteinMPNN generation
Removed old raw ProteinMPNN file: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/conditional_probs/raw_ProteinMPNN_outputs/UBE4B_multi/proteinmpnn/conditional_probs_only/UBE4B_multi.npz

RUN: python "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/ProteinMPNN/protein_mpnn_run.py" --pdb_path "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/pdbs/UBE4B_mu

RuntimeError: Command failed with exit code 1:
python -m kermut.cmdline.preprocess_data.extract_esm2_zero_shots dataset=multi +multi.use_id=true +multi.id=UBE4B_multi

Log:

  0%|          | 0/1 [00:00<?, ?it/s]--- Computing zero-shots for UBE4B_multi (1/1) ---

  0%|          | 0/1 [00:00<?, ?it/s]
Error executing job with overrides: ['dataset=multi', '+multi.use_id=true', '+multi.id=UBE4B_multi']
Traceback (most recent call last):
  File "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/kermut/cmdline/preprocess_data/extract_esm2_zero_shots.py", line 145, in extract_esm2_zero_shots
    raise FileNotFoundError(f"DMS assay file not found: {file_in}")
FileNotFoundError: DMS assay file not found: data/cv_folds_singles_substitutions/UBE4B_multi.csv

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.


In [ ]:
# Extract ESM-2 zero-shot scores

if RUN_ESM2_ZERO_SHOT:
    if MUTATION_LEVEL == "multi" and ZERO_SHOT_MODE != "default":
        raise ValueError(
            "For the multi-mutant Kermut analysis, use ZERO_SHOT_MODE='default'. "
            "Zero-shot ensemble replacement is intentionally disabled for the multi run."
        )

    for ds in DATASETS:
        dms_id = ds["DMS_id"]

        print("\n" + "=" * 120)
        print(f"{dms_id}: ESM-2 zero-shot extraction")
        print("=" * 120)

        if FORCE_RERUN_BENCHMARK or OVERWRITE:
            for pattern in [f"{dms_id}*", f"*{dms_id}*"]:
                for file in KERMUT_ZERO_SHOT_DIR.glob(pattern):
                    if file.is_file():
                        file.unlink()
                        print("Removed old zero-shot file:", file)

        cmd = (
            "python -m kermut.cmdline.preprocess_data.extract_esm2_zero_shots "
            f"dataset={KERMUT_DATASET_CLI} "
            "data.embedding.mode=multiples "
            f"{hydra_dataset_id_overrides(dms_id)}"
        )

        log_file = LOG_DIR / f"{dms_id}_extract_esm2_zero_shots.log"

        run_cmd(
            cmd,
            cwd=KERMUT_REPO,
            env=env,
            log_file=log_file,
        )

else:
    print("RUN_ESM2_ZERO_SHOT=False -> skipped ESM-2 zero-shot extraction.")


UBE4B_multi: ESM-2 zero-shot extraction

RUN: python -m kermut.cmdline.preprocess_data.extract_esm2_zero_shots dataset=multi data.embedding.mode=multiples +multi.use_id=true +multi.id=UBE4B_multi

  0%|          | 0/1 [00:00<?, ?it/s]--- Computing zero-shots for UBE4B_multi (1/1) ---

100%|██████████| 1/1 [35:42<00:00, 2142.46s/it]


DLG4_abundance_multi: ESM-2 zero-shot extraction

RUN: python -m kermut.cmdline.preprocess_data.extract_esm2_zero_shots dataset=multi data.embedding.mode=multiples +multi.use_id=true +multi.id=DLG4_abundance_multi

  0%|          | 0/1 [00:00<?, ?it/s]--- Computing zero-shots for DLG4_abundance_multi (1/1) ---

100%|██████████| 1/1 [00:14<00:00, 14.64s/it]


DLG4_binding_multi: ESM-2 zero-shot extraction

RUN: python -m kermut.cmdline.preprocess_data.extract_esm2_zero_shots dataset=multi data.embedding.mode=multiples +multi.use_id=true +multi.id=DLG4_binding_multi

  0%|          | 0/1 [00:00<?, ?it/s]--- Computing zero-shots for DLG4_binding_multi (1/1) 

In [ ]:
# Check preprocessing output files

preprocess_check_rows = []

for ds in DATASETS:
    dms_id = ds["DMS_id"]

    possible_files = {
        "embedding_h5": KERMUT_EMB_DIR / f"{dms_id}.h5",
        "coords_npy": KERMUT_COORDS_DIR / f"{dms_id}.npy",
        "proteinmpnn_npy": KERMUT_PROC_MPNN_DIR / f"{dms_id}.npy",
        "zero_shot_csv": KERMUT_ZERO_SHOT_DIR / f"{dms_id}.csv",
        "kermut_csv": KERMUT_CV_DIR / f"{dms_id}.csv",
    }

    row = {"DMS_id": dms_id}

    for key, path in possible_files.items():
        row[f"{key}_exists"] = path.exists()
        row[f"{key}_path"] = str(path)
        row[f"{key}_size_MB"] = path.stat().st_size / 1024 / 1024 if path.exists() else np.nan

    preprocess_check_rows.append(row)

preprocess_check_df = pd.DataFrame(preprocess_check_rows)

display(
    preprocess_check_df[
        [
            "DMS_id",
            "kermut_csv_exists",
            "embedding_h5_exists",
            "coords_npy_exists",
            "proteinmpnn_npy_exists",
            "zero_shot_csv_exists",
            "embedding_h5_size_MB",
            "coords_npy_size_MB",
            "proteinmpnn_npy_size_MB",
            "zero_shot_csv_size_MB",
        ]
    ]
)

,DMS_id,kermut_csv_exists,embedding_h5_exists,coords_npy_exists,proteinmpnn_npy_exists,zero_shot_csv_exists,embedding_h5_size_MB,coords_npy_size_MB,proteinmpnn_npy_size_MB,zero_shot_csv_size_MB
0,UBE4B_multi,True,True,True,True,True,431.296066,0.013546,0.089615,103.836516
1,DLG4_abundance_multi,True,True,True,True,True,28.088867,0.001083,0.006531,0.763030
2,DLG4_binding_multi,True,True,True,True,True,33.549225,0.001083,0.006531,0.912275
3,GRB2_multi,True,True,True,True,True,307.214783,0.002605,0.016678,16.522595


In [ ]:
# Optional zero-shot replacement

if ZERO_SHOT_MODE == "ensemble":
    if MUTATION_LEVEL == "multi":
        raise ValueError(
            "ZERO_SHOT_MODE='ensemble' is disabled for multi-mutant Kermut. "
            "Use ZERO_SHOT_MODE='default'."
        )

    print("Replacing original ESM-2 zero-shot scores with ensemble scores.")

    for ds in DATASETS:
        dms_id = ds["DMS_id"]

        if "zero_shot_ensemble_path" not in ds:
            raise ValueError(
                f"{dms_id}: ZERO_SHOT_MODE='ensemble', but dataset has no "
                "'zero_shot_ensemble_path'."
            )

        src = Path(ds["zero_shot_ensemble_path"])
        dst = KERMUT_ZERO_SHOT_DIR / f"{dms_id}.csv"

        if not src.exists():
            raise FileNotFoundError(f"{dms_id}: zero-shot ensemble file not found: {src}")

        df_zs = pd.read_csv(src)

        if "mutant" not in df_zs.columns:
            raise ValueError(f"{dms_id}: zero-shot ensemble file has no mutant column: {src}")

        if "score" not in df_zs.columns and "y_pred" in df_zs.columns:
            df_zs = df_zs.rename(columns={"y_pred": "score"})

        if "score" not in df_zs.columns:
            raise ValueError(
                f"{dms_id}: zero-shot ensemble file needs score or y_pred column. "
                f"Columns: {df_zs.columns.tolist()}"
            )

        df_zs[["mutant", "score"]].to_csv(dst, index=False)

        print(f"{dms_id}: wrote ensemble zero-shot scores to {dst}")

else:
    print("ZERO_SHOT_MODE='default' -> using original Kermut ESM-2 zero-shot scores.")

ZERO_SHOT_MODE='default' -> using original Kermut ESM-2 zero-shot scores.


In [ ]:
# Run Kermut benchmark

RUN_KERMUT_BENCHMARK = True
if RUN_KERMUT_BENCHMARK:
    for ds in DATASETS:
        dms_id = ds["DMS_id"]

        print("\n" + "=" * 120)
        print(f"{dms_id}: Kermut benchmark")
        print("Kernel:", KERNEL_NAME)
        print("Dataset CLI:", KERMUT_DATASET_CLI)
        print("=" * 120)

        if FORCE_RERUN_BENCHMARK or OVERWRITE:
            if TEMP_PRED_DIR.exists():
                for pattern in [f"{dms_id}*", f"*{dms_id}*"]:
                    for file in TEMP_PRED_DIR.rglob(pattern):
                        if file.is_file():
                            file.unlink()

        cmd = (
            "python proteingym_benchmark.py --multirun "
            f"dataset={KERMUT_DATASET_CLI} "
            "data.paths.DMS_input_folder=data/cv_folds_multiples_substitutions "
            "data.paths.embeddings_singles=data/embeddings/substitutions_multiples/ESM2 "
            "data.embedding.mode=multiples "
            f"{hydra_dataset_id_overrides(dms_id)} "
            "cv_scheme=split_esm2 "
            f"kernel={KERNEL_NAME}"
        )

        log_file = LOG_DIR / f"{dms_id}_kermut_benchmark_{KERNEL_NAME}.log"

        run_cmd(
            cmd,
            cwd=KERMUT_REPO,
            env=env,
            log_file=log_file,
        )

else:
    print("RUN_KERMUT_BENCHMARK=False -> skipped Kermut benchmark.")

# GRB2 und UBE4B subsamplen (too much variants for Kermut)

In [ ]:
# Create multi-subsample dataset for Kermut
# Keeps all test variants, samples N train+val variants
# Works for GRB2_multi and UBE4B_multi


from pathlib import Path
import pandas as pd
import shutil

KERMUT_REPO = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut"
)

DATA_DIR = KERMUT_REPO / "data"

DMS_DIR = DATA_DIR / "cv_folds_multiples_substitutions"
REF_FILE = DATA_DIR / "DMS_substitutions.csv"

# Choose dataset here
SOURCE_ID = "UBE4B_multi"
SUB_ID = "UBE4B_multi_sub10000"
# Alternative:
# SOURCE_ID = "GRB2_multi"
# SUB_ID = "GRB2_multi_sub10000"

N_TRAINVAL = 10000
RANDOM_STATE = 37

# Create sampled CSV
source_csv = DMS_DIR / f"{SOURCE_ID}.csv"
sub_csv = DMS_DIR / f"{SUB_ID}.csv"

if not source_csv.exists():
    raise FileNotFoundError(source_csv)

df = pd.read_csv(source_csv)

if "split_esm2" not in df.columns:
    raise ValueError(f"'split_esm2' column not found. Columns: {df.columns.tolist()}")

trainval_df = df[df["split_esm2"].astype(str).isin(["train", "val"])].copy()
test_df = df[df["split_esm2"].astype(str).eq("test")].copy()

print("=" * 100)
print("SOURCE:", SOURCE_ID)
print("=" * 100)
print("Original total:", len(df))
print("Original split counts:")
print(df["split_esm2"].value_counts(dropna=False))

if len(trainval_df) == 0:
    raise ValueError(f"{SOURCE_ID}: no train/val rows found.")

if len(test_df) == 0:
    raise ValueError(f"{SOURCE_ID}: no test rows found.")

if len(trainval_df) > N_TRAINVAL:
    trainval_sub = trainval_df.sample(
        n=N_TRAINVAL,
        random_state=RANDOM_STATE,
        replace=False,
    ).copy()
else:
    trainval_sub = trainval_df.copy()

sub_df = pd.concat([trainval_sub, test_df], ignore_index=True)

# Keep original split labels: train/val/test.
# Kermut trains on sampled train+val and evaluates on full test.
sub_df.to_csv(sub_csv, index=False)

print("\n" + "=" * 100)
print("SUBSAMPLE:", SUB_ID)
print("=" * 100)
print("Written:", sub_csv)
print("Subsample total:", len(sub_df))
print("Subsample split counts:")
print(sub_df["split_esm2"].value_counts(dropna=False))
print("train+val:", sub_df["split_esm2"].astype(str).isin(["train", "val"]).sum())
print("test:", sub_df["split_esm2"].astype(str).eq("test").sum())

print("\nFirst rows:")
display(sub_df.head())

SOURCE: UBE4B_multi
Original total: 87435
Original split counts:
split_esm2
train    69948
test      8744
val       8743
Name: count, dtype: int64

SUBSAMPLE: UBE4B_multi_sub10000
Written: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/cv_folds_multiples_substitutions/UBE4B_multi_sub10000.csv
Subsample total: 18744
Subsample split counts:
split_esm2
train    8894
test     8744
val      1106
Name: count, dtype: int64
train+val: 10000
test: 8744

First rows:


,mutant,DMS_score,mutated_sequence,split_esm2
0,R1104G:S1171R,-2.416596,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,train
1,E1093G:F1141S,-1.781984,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,val
2,Q1144L:D1172N,-1.290177,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,train
3,N1089I:D1109V,-0.419538,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,train
4,K1088M:F1103L,-1.407633,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,train


In [ ]:
# Add subsample dataset to DMS_substitutions.csv
# Works for GRB2_multi_sub10000 and UBE4B_multi_sub10000

ref = pd.read_csv(REF_FILE)

if SOURCE_ID not in set(ref["DMS_id"].astype(str)):
    raise ValueError(f"{SOURCE_ID} not found in reference file.")

source_row = ref[ref["DMS_id"].astype(str).eq(SOURCE_ID)].iloc[0].copy()
source_row["DMS_id"] = SUB_ID

# Update count columns if present
if "DMS_total_number_mutants" in ref.columns:
    source_row["DMS_total_number_mutants"] = len(sub_df)

if "DMS_number_single_mutants" in ref.columns:
    source_row["DMS_number_single_mutants"] = 0

if "includes_multiple_mutants" in ref.columns:
    source_row["includes_multiple_mutants"] = True

# Remove old sub row if it already exists
ref = ref[~ref["DMS_id"].astype(str).eq(SUB_ID)].copy()

ref = pd.concat([ref, pd.DataFrame([source_row])], ignore_index=True)
ref.to_csv(REF_FILE, index=False)

print("Updated reference file:", REF_FILE)
display(ref[ref["DMS_id"].astype(str).isin([SOURCE_ID, SUB_ID])])

Updated reference file: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/DMS_substitutions.csv


,DMS_index,DMS_id,DMS_filename,UniProt_ID,taxon,source_organism,target_seq,seq_len,includes_multiple_mutants,DMS_total_number_mutants,...,raw_DMS_filename,raw_DMS_phenotype_name,raw_DMS_directionality,raw_DMS_mutant_column,weight_file_name,pdb_file,pdb_range,ProteinGym_version,raw_mut_offset,coarse_selection_type
241,NaN,UBE4B_multi,UBE4B_multi.csv,UBE4B_multi,NaN,NaN,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,NaN,True,87435,...,UBE4B_multi.csv,NaN,NaN,NaN,NaN,UBE4B_multi.pdb,NaN,NaN,NaN,NaN
246,NaN,UBE4B_multi_sub10000,UBE4B_multi.csv,UBE4B_multi,NaN,NaN,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,NaN,True,18744,...,UBE4B_multi.csv,NaN,NaN,NaN,NaN,UBE4B_multi.pdb,NaN,NaN,NaN,NaN


In [ ]:
# Copy feature files from full multi dataset to subsample dataset name

copy_pairs = [
    # PDB
    (
        DATA_DIR / "structures" / "pdbs" / f"{SOURCE_ID}.pdb",
        DATA_DIR / "structures" / "pdbs" / f"{SUB_ID}.pdb",
    ),

    # Coordinates
    (
        DATA_DIR / "structures" / "coords" / f"{SOURCE_ID}.npy",
        DATA_DIR / "structures" / "coords" / f"{SUB_ID}.npy",
    ),

    # ProteinMPNN processed conditional probabilities
    (
        DATA_DIR / "conditional_probs" / "ProteinMPNN" / f"{SOURCE_ID}.npy",
        DATA_DIR / "conditional_probs" / "ProteinMPNN" / f"{SUB_ID}.npy",
    ),

    # Raw ProteinMPNN output, not always needed by benchmark, but useful for consistency
    (
        DATA_DIR / "conditional_probs" / "raw_ProteinMPNN_outputs" / SOURCE_ID / "proteinmpnn" / "conditional_probs_only" / f"{SOURCE_ID}.npz",
        DATA_DIR / "conditional_probs" / "raw_ProteinMPNN_outputs" / SUB_ID / "proteinmpnn" / "conditional_probs_only" / f"{SUB_ID}.npz",
    ),

    # ESM-2 embeddings for multiple mutants
    (
        DATA_DIR / "embeddings" / "substitutions_multiples" / "ESM2" / f"{SOURCE_ID}.h5",
        DATA_DIR / "embeddings" / "substitutions_multiples" / "ESM2" / f"{SUB_ID}.h5",
    ),

    # ESM-2 zero-shot scores
    (
        DATA_DIR / "zero_shot_fitness_predictions" / "ESM2" / "650M" / f"{SOURCE_ID}.csv",
        DATA_DIR / "zero_shot_fitness_predictions" / "ESM2" / "650M" / f"{SUB_ID}.csv",
    ),
]

for src, dst in copy_pairs:
    if not src.exists():
        print("[WARN] missing source, skipped:")
        print(" ", src)
        continue

    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

    print("Copied:")
    print(" ", src)
    print(" ->", dst)

print("\nFeature copy finished for:")
print("SOURCE_ID:", SOURCE_ID)
print("SUB_ID:", SUB_ID)

Copied:
  /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/pdbs/UBE4B_multi.pdb
 -> /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/pdbs/UBE4B_multi_sub10000.pdb
Copied:
  /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/coords/UBE4B_multi.npy
 -> /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/structures/coords/UBE4B_multi_sub10000.npy
Copied:
  /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/conditional_probs/ProteinMPNN/UBE4B_multi.npy
 -> /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/conditional_probs/ProteinMPNN/UBE4B_multi_sub10000.npy
Copied:
  /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/conditional_probs/raw_ProteinMPNN_outputs/UBE4B_multi/proteinmpnn/conditional_probs_only/UBE4B_multi.npz
 -> /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/data/conditional_probs/raw_ProteinMPNN_outputs/UB

In [ ]:
# Run Kermut benchmark for subsampled multi dataset

RUN_KERMUT_BENCHMARK = True

TARGET_DMS_ID = SUB_ID

KERMUT_RUN_DATASETS = [
    {
        "DMS_id": TARGET_DMS_ID,
        "pretty": TARGET_DMS_ID,
        "folder": SOURCE_ID.replace("_multi", ""),
    }
]

if RUN_KERMUT_BENCHMARK:
    for ds in KERMUT_RUN_DATASETS:
        dms_id = ds["DMS_id"]

        print("\n" + "=" * 120)
        print(f"{dms_id}: Kermut benchmark")
        print("Kernel:", KERNEL_NAME)
        print("Dataset CLI:", KERMUT_DATASET_CLI)
        print("=" * 120)

        if FORCE_RERUN_BENCHMARK or OVERWRITE:
            if TEMP_PRED_DIR.exists():
                for pattern in [f"{dms_id}*", f"*{dms_id}*"]:
                    for file in TEMP_PRED_DIR.rglob(pattern):
                        if file.is_file():
                            file.unlink()

        cmd = (
            "python proteingym_benchmark.py --multirun "
            f"dataset={KERMUT_DATASET_CLI} "
            "data.paths.DMS_input_folder=data/cv_folds_multiples_substitutions "
            "data.paths.embeddings_singles=data/embeddings/substitutions_multiples/ESM2 "
            "data.embedding.mode=multiples "
            f"+multi.use_id=true +multi.id={dms_id} "
            "cv_scheme=split_esm2 "
            f"kernel={KERNEL_NAME}"
        )

        log_file = LOG_DIR / f"{dms_id}_kermut_benchmark_{KERNEL_NAME}.log"

        print("Command:")
        print(cmd)
        print("Log file:")
        print(log_file)

        run_cmd(
            cmd,
            cwd=KERMUT_REPO,
            env=env,
            log_file=log_file,
        )

else:
    print("RUN_KERMUT_BENCHMARK=False -> skipped Kermut benchmark.")


UBE4B_multi_sub10000: Kermut benchmark
Kernel: kermut
Dataset CLI: multi
Command:
python proteingym_benchmark.py --multirun dataset=multi data.paths.DMS_input_folder=data/cv_folds_multiples_substitutions data.paths.embeddings_singles=data/embeddings/substitutions_multiples/ESM2 data.embedding.mode=multiples +multi.use_id=true +multi.id=UBE4B_multi_sub10000 cv_scheme=split_esm2 kernel=kermut
Log file:
/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/Kermut/variant_runs/multi_original_default/logs/UBE4B_multi_sub10000_kermut_benchmark_kermut.log

RUN: python proteingym_benchmark.py --multirun dataset=multi data.paths.DMS_input_folder=data/cv_folds_multiples_substitutions data.paths.embeddings_singles=data/embeddings/substitutions_multiples/ESM2 data.embedding.mode=multiples +multi.use_id=true +multi.id=UBE4B_multi_sub10000 cv_scheme=split_esm2 kernel=kermut
[2026-06-08 13:12:21,451][HYDRA] Launching 1 jobs locally
[2026-06-08 13:12:21,452][HYDRA] 	#0 : dataset=mul

In [ ]:
# Evaluate Kermut multi-mutant benchmark predictions
# Metrics: Spearman, Pearson, R2, RMSE, MAE, NDCG@10%

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, ndcg_score


KERMUT_REPO = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut"
)

KERMUT_OUTPUT_DIR = KERMUT_REPO / "outputs" / "split_esm2" / "kermut"

FINAL_BASE = Path(
    "/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data"
)

OUT_DIR = FINAL_BASE / "multi_summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS_OUT = OUT_DIR / "multi_kermut_metrics.tsv"
PREDICTIONS_OUT = OUT_DIR / "multi_kermut_predictions_long.tsv"

KERMUT_MULTI_DATASETS = [
    {
        "DMS_id": "UBE4B_multi_sub10000",
        "pretty": "UBE4B",
    },
    {
        "DMS_id": "DLG4_abundance_multi",
        "pretty": "DLG4 abundance",
    },
    {
        "DMS_id": "DLG4_binding_multi",
        "pretty": "DLG4 binding",
    },
    {
        "DMS_id": "GRB2_multi_sub10000",
        "pretty": "GRB2",
    },
]

TOP_FRAC = 0.10

def make_nonnegative_relevance(y_true):
    y_true = np.asarray(y_true, dtype=float)
    relevance = y_true - np.nanmin(y_true)
    relevance = np.clip(relevance, 0.0, None)
    return relevance


def ndcg_at_fraction(y_true, y_score, frac=0.10):
    y_true = pd.to_numeric(pd.Series(y_true), errors="coerce").to_numpy(float)
    y_score = pd.to_numeric(pd.Series(y_score), errors="coerce").to_numpy(float)

    mask = np.isfinite(y_true) & np.isfinite(y_score)
    y_true = y_true[mask]
    y_score = y_score[mask]

    if len(y_true) < 3:
        return np.nan

    relevance = make_nonnegative_relevance(y_true)

    if np.max(relevance) == 0:
        return np.nan

    k = max(1, int(np.ceil(frac * len(y_true))))

    return float(
        ndcg_score(
            relevance.reshape(1, -1),
            y_score.reshape(1, -1),
            k=k,
        )
    )


def compute_metrics(y_true, y_pred, frac=0.10):
    y_true = pd.to_numeric(pd.Series(y_true), errors="coerce").to_numpy(float)
    y_pred = pd.to_numeric(pd.Series(y_pred), errors="coerce").to_numpy(float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    out = {
        "n": int(len(y_true)),
        "spearman": np.nan,
        "pearson": np.nan,
        "r2": np.nan,
        "rmse": np.nan,
        "mae": np.nan,
        "ndcg_10": np.nan,
        "y_true_mean": np.nan,
        "y_true_std": np.nan,
        "y_pred_mean": np.nan,
        "y_pred_std": np.nan,
    }

    if len(y_true) < 3:
        return out

    out["y_true_mean"] = float(np.mean(y_true))
    out["y_true_std"] = float(np.std(y_true))
    out["y_pred_mean"] = float(np.mean(y_pred))
    out["y_pred_std"] = float(np.std(y_pred))

    if np.std(y_true) > 0 and np.std(y_pred) > 0:
        out["spearman"] = float(spearmanr(y_true, y_pred).correlation)
        out["pearson"] = float(pearsonr(y_true, y_pred)[0])

    out["r2"] = float(r2_score(y_true, y_pred))
    out["rmse"] = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    out["mae"] = float(mean_absolute_error(y_true, y_pred))
    out["ndcg_10"] = ndcg_at_fraction(y_true, y_pred, frac=frac)

    return out


# Load Kermut outputs and evaluate

all_metrics = []
all_predictions = []

for ds in KERMUT_MULTI_DATASETS:
    dms_id = ds["DMS_id"]
    pretty = ds["pretty"]

    pred_path = KERMUT_OUTPUT_DIR / f"{dms_id}.csv"

    print("\n" + "=" * 120)
    print(dms_id)
    print("=" * 120)
    print("Prediction file:", pred_path)

    if not pred_path.exists():
        print("[WARN] Missing prediction file, skipped.")
        continue

    df = pd.read_csv(pred_path)

    required_cols = {"mutant", "fold", "y", "y_pred"}
    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(
            f"{dms_id}: missing columns {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )

    print("Raw shape:", df.shape)
    print("Fold counts:")
    print(df["fold"].value_counts(dropna=False))

    # Kermut output contains predictions only for test rows;
    # train rows usually remain NaN in y/y_pred.
    eval_df = df.dropna(subset=["y", "y_pred"]).copy()

    print("Evaluated rows:", len(eval_df))
    print("Example rows:")
    display(eval_df.head())

    metrics = compute_metrics(
        eval_df["y"],
        eval_df["y_pred"],
        frac=TOP_FRAC,
    )

    row = {
        "dataset": dms_id,
        "pretty": pretty,
        "model": "Kermut",
        "split": "test",
        **metrics,
    }

    all_metrics.append(row)

    eval_df["dataset"] = dms_id
    eval_df["pretty"] = pretty
    eval_df["model"] = "Kermut"
    eval_df["split"] = "test"

    all_predictions.append(eval_df)

metrics_df = pd.DataFrame(all_metrics)

if len(all_predictions) > 0:
    predictions_df = pd.concat(all_predictions, ignore_index=True)
else:
    predictions_df = pd.DataFrame()

print("\nKermut multi metrics:")
display(metrics_df)

metrics_df.to_csv(METRICS_OUT, sep="\t", index=False)
predictions_df.to_csv(PREDICTIONS_OUT, sep="\t", index=False)

print("Saved metrics to:")
print(METRICS_OUT)

print("Saved predictions to:")
print(PREDICTIONS_OUT)


UBE4B_multi_sub10000
Prediction file: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/outputs/split_esm2/kermut/UBE4B_multi_sub10000.csv
Raw shape: (18744, 5)
Fold counts:
fold
NaN     10000
test     8744
Name: count, dtype: int64
Evaluated rows: 8744
Example rows:


,mutant,fold,y,y_pred,y_var
10000,M1108R:E1152D:H1173L,test,-1.403080,-0.430184,0.528690
10001,K1076I:V1082L,test,0.681643,0.176226,0.525958
10002,P1137S:K1168N:D1172Y,test,1.461210,0.108779,0.524883
10003,A1090P:T1110S:Q1160L,test,-0.681126,-0.198248,0.532425
10004,L1078F:V1086L:R1166T,test,0.862360,0.380711,0.533872



DLG4_abundance_multi
Prediction file: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/outputs/split_esm2/kermut/DLG4_abundance_multi.csv
Raw shape: (5696, 5)
Fold counts:
fold
NaN     5126
test     570
Name: count, dtype: int64
Evaluated rows: 570
Example rows:


,mutant,fold,y,y_pred,y_var
5126,V5R:G35K,test,-0.883566,-0.803973,0.252141
5127,S10I:S61G,test,0.845770,0.527880,0.243626
5128,S10I:G54I,test,-0.612068,-0.536655,0.227052
5129,S10I:E42S,test,1.663331,1.034257,0.238430
5130,G41A:I49L,test,0.482608,0.663860,0.268217



DLG4_binding_multi
Prediction file: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/outputs/split_esm2/kermut/DLG4_binding_multi.csv
Raw shape: (6810, 5)
Fold counts:
fold
NaN     6129
test     681
Name: count, dtype: int64
Evaluated rows: 681
Example rows:


...


GRB2_multi_sub10000
Prediction file: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/Kermut/outputs/split_esm2/kermut/GRB2_multi_sub10000.csv
Raw shape: (16234, 5)
Fold counts:
fold
NaN     10000
test     6234
Name: count, dtype: int64
Evaluated rows: 6234
Example rows:


,mutant,fold,y,y_pred,y_var
10000,D166G:G176V,test,-0.565435,-0.557446,0.229204
10001,R179P:M186E,test,0.248675,-0.206876,0.213332
10002,G173R:T202R,test,-0.374592,0.214889,0.238957
10003,L175V:V213W,test,0.576081,0.324723,0.225989
10004,G180L:V213W,test,0.037462,0.100681,0.223338



Kermut multi metrics:


,dataset,pretty,model,split,n,spearman,pearson,r2,rmse,mae,ndcg_10,y_true_mean,y_true_std,y_pred_mean,y_pred_std
0,UBE4B_multi_sub10000,UBE4B,Kermut,test,8744,0.659431,0.683854,0.467533,0.723762,0.553075,0.880572,-0.011917,0.991857,-0.001210,0.680737
1,DLG4_abundance_multi,DLG4 abundance,Kermut,test,570,0.829871,0.850742,0.722053,0.537824,0.417715,0.916256,-0.001947,1.020138,0.028305,0.838493
2,DLG4_binding_multi,DLG4 binding,Kermut,test,681,0.863370,0.892946,0.796410,0.459292,0.356710,0.957436,0.043651,1.017914,0.040107,0.877893
3,GRB2_multi_sub10000,GRB2,Kermut,test,6234,0.855802,0.857663,0.734672,0.525464,0.415925,0.949290,0.041208,1.020120,0.022190,0.899201


Saved metrics to:
/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/multi_summary/multi_kermut_metrics.tsv
Saved predictions to:
/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/multi_summary/multi_kermut_predictions_long.tsv


In [ ]:
# Display Kermut multi-mutant metric summary

if metrics_df.empty:
    print("No metrics available.")
else:
    display_cols = [
        "pretty",
        "model",
        "split",
        "n",
        "spearman",
        "ndcg_10",
        "r2",
        "pearson",
        "rmse",
        "mae",
        "y_true_mean",
        "y_true_std",
        "y_pred_mean",
        "y_pred_std",
    ]

    print("Kermut multi-mutant metrics:")
    display(
        metrics_df[display_cols]
        .sort_values(["pretty", "model"])
        .reset_index(drop=True)
    )

    print("\nSpearman pivot:")
    display(
        metrics_df.pivot_table(
            index="pretty",
            columns="model",
            values="spearman",
            aggfunc="mean",
        )
    )

    print("\nNDCG@10% pivot:")
    display(
        metrics_df.pivot_table(
            index="pretty",
            columns="model",
            values="ndcg_10",
            aggfunc="mean",
        )
    )

    print("\nR2 pivot:")
    display(
        metrics_df.pivot_table(
            index="pretty",
            columns="model",
            values="r2",
            aggfunc="mean",
        )
    )

Kermut multi-mutant metrics:


,pretty,model,split,n,spearman,ndcg_10,r2,pearson,rmse,mae,y_true_mean,y_true_std,y_pred_mean,y_pred_std
0,DLG4 abundance,Kermut,test,570,0.829871,0.916256,0.722053,0.850742,0.537824,0.417715,-0.001947,1.020138,0.028305,0.838493
1,DLG4 binding,Kermut,test,681,0.863370,0.957436,0.796410,0.892946,0.459292,0.356710,0.043651,1.017914,0.040107,0.877893
2,GRB2,Kermut,test,6234,0.855802,0.949290,0.734672,0.857663,0.525464,0.415925,0.041208,1.020120,0.022190,0.899201
3,UBE4B,Kermut,test,8744,0.659431,0.880572,0.467533,0.683854,0.723762,0.553075,-0.011917,0.991857,-0.001210,0.680737



Spearman pivot:


model,Kermut
pretty,
DLG4 abundance,0.829871
DLG4 binding,0.863370
GRB2,0.855802
UBE4B,0.659431



NDCG@10% pivot:


model,Kermut
pretty,
DLG4 abundance,0.916256
DLG4 binding,0.957436
GRB2,0.949290
UBE4B,0.880572



R2 pivot:


model,Kermut
pretty,
DLG4 abundance,0.722053
DLG4 binding,0.796410
GRB2,0.734672
UBE4B,0.467533
